# Best-of-N Speed Benchmark Across Quantization Levels

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple quantization settings, such as fp16 and
GPTQ-int4 model variants.

Each configuration is loaded with vLLM, warmed up, timed for
num_trials runs, and then unloaded before the next configuration is
tested. This keeps the comparison focused on how quantization affects
throughput under the same benchmark settings.

Use this notebook to compare speed across precision or quantization
levels. Use benchmark_speed_bon_models_v1.ipynb for the separate
model-axis sweep at a fixed precision.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

import gc
import statistics
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1
from utils.load_data import load_data_hf

In [2]:
# Dataset path
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

In [3]:
# Quantization configs to benchmark.
# GPTQ requires a pre-quantized model directory (point model_dir to it).
def _cfg(name, subdir, quantization, dtype):
    return {
        "name":         name,
        "model_dir":    os.path.join(base_dir, subdir),
        "quantization": quantization,
        "load_format":  "auto",
        "dtype":        dtype,
    }


quant_configs = [
    _cfg("llama-3b fp16",     "Llama3.2-3B-Instruct",          None,   "float16"),
    _cfg("llama-3b gptq",     "Llama3.2-3B-Instruct-GPTQ",     "gptq", "auto"),
    _cfg("qwen-3b fp16",      "Qwen2.5-3B-Instruct",           None,   "float16"),
    _cfg("qwen-3b gptq-int4", "Qwen2.5-3B-Instruct-GPTQ-Int4", "gptq", "auto"),
    _cfg("qwen-7b fp16",      "Qwen2.5-7B-Instruct",           None,   "float16"),
    _cfg("qwen-7b gptq-int4", "Qwen2.5-7B-Instruct-GPTQ-Int4", "gptq", "auto"),
]

In [4]:
# Best-of-N search params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 256
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 5                  # cap on questions per benchmark
num_trials = 2                     # timed runs per config
warmup = 1                         # untimed warmup runs per config
llm_gpu_memory_utilization = 0.5

In [5]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 5


## Helpers

In [6]:
def gpu_mem_used_gb(device=0):
    """Driver-level used GPU memory; sees both PyTorch and vLLM allocs."""
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


def benchmark_quant(qcfg, config, prompts, num_trials, warmup=1):
    """Load `qcfg["model_dir"]` under vLLM with `qcfg`'s quantization /
    dtype / load_format, warm up, time `num_trials` runs of
    best_of_n_v1, then tear down. Returns (name, gpu_mem_gb, trial_times).
    """
    print(f"\n=== {qcfg['name']} ===")

    # enforce_eager=True disables CUDA graphs - skips cudagraph capture cost
    # on every model load, giving more stable latency at small num_trials.
    llm = LLM(
        model=qcfg["model_dir"],
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype=qcfg["dtype"],
        quantization=qcfg["quantization"],
        load_format=qcfg["load_format"],
        seed=config.seed,
    )
    gc.collect()
    torch.cuda.empty_cache()
    gpu_mem = gpu_mem_used_gb()
    print(f"  GPU memory used: {gpu_mem:.2f} GB")

    # Warmup (untimed) - absorbs first-call init inside best_of_n_v1
    for w in range(warmup):
        bon_search_v1.best_of_n_v1(prompts, config, llm, 10_000 + w)

    times = []
    for trial_idx in range(num_trials):
        start = time.perf_counter()
        bon_search_v1.best_of_n_v1(prompts, config, llm, trial_idx)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        print(
            f"  trial {trial_idx}: {elapsed:>7.2f}s total, "
            f"{elapsed / len(prompts):.4f}s/question"
        )

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    return qcfg["name"], gpu_mem, times

## Run benchmark

One config at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [ ]:
results = []
for qcfg in quant_configs:
    name, mem, times = benchmark_quant(
        qcfg, config, batch_of_questions, num_trials, warmup=warmup,
    )
    results.append((name, mem, times))


=== llama-3b fp16 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.64s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.75s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 16.53 GB
  trial 0:  303.21s total, 60.6417s/question
  trial 1:  291.90s total, 58.3801s/question


[rank0]:[W612 18:44:24.682559277 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== llama-3b gptq ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.81it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.81it/s]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 16.63 GB


## Summary

In [ ]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'quantization':<25}{'gpu (GB)':>10}"
    f"{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, mem, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<25}{mem:>10.2f}"
        f"{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )